## 1. Setup

In [ ]:
import sys
sys.path.append("..")

from pathlib import Path

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from src.paths import MANUAL_PATH, PROCESSED_PATH
from src.consumption import (
    OVERRIDE_COLUMNS,
    extract_merchant_text,
    classify_consumption_transaction,
    classify_fixed_variable_from_category,
    preview_unique,
)

## 2. Paths & Load Data

In [ ]:
PROJECT_ROOT = Path("..")
DATA_PATH = PROCESSED_PATH

transactions = pd.read_csv(
    DATA_PATH / "transactions_clean.csv"
)

transactions.columns.tolist()

## 3. Prepare Consumption Dataset

In [ ]:
df = transactions.copy()

df["date"] = pd.to_datetime(df["date"])
df["month"] = df["date"].dt.to_period("M").dt.to_timestamp()

consumption = df[
    df["economic_type"] == "Consumption"
].copy()

consumption["spending_amount"] = consumption["amount"].abs()

consumption.head()

## 4. Detail Column Audit

In [ ]:
candidate_detail_cols = [
    "name",
    "description",
    "mcc_code",
    "type",
    "category",
    "original_amount",
    "original_currency"
]

available_detail_cols = [
    col for col in candidate_detail_cols
    if col in consumption.columns
]

available_detail_cols

## 5. Merchant Classification

Layered, first match wins (see `src/consumption.py`):

1. **Private overrides** – `data/manual/merchant_overrides_manual.csv` (gitignored), for merchant-specific corrections
2. **Strong MCC** – merchant category code set by the card network, the most reliable signal
3. **Keyword rules** – generic name patterns, mainly for direct debits (no MCC)
4. **Weak MCC** – generic codes (e.g. 5499, 5999) as fallback only
5. Otherwise `Uncategorized`

### 5.1 Merchant text

In [ ]:
consumption["merchant_raw"] = consumption["name"].fillna("Unknown")

merchant_text = consumption.apply(
    lambda row: extract_merchant_text(row["name"], row["description"]),
    axis=1,
    result_type="expand"
)

consumption["merchant_clean"] = merchant_text[0]
consumption["merchant_source"] = merchant_text[1]

print(consumption["merchant_source"].value_counts())

consumption[
    [
        "date",
        "merchant_raw",
        "description",
        "merchant_clean",
        "merchant_source",
        "mcc_code",
        "spending_amount"
    ]
].head(30)

### 5.2 Private overrides

In [ ]:
# Example seed only -- the real overrides live in the gitignored CSV and
# are never overwritten once it exists.
merchant_overrides_example_seed = pd.DataFrame(
    [
        {
            "pattern": r"EXAMPLE PRACTICE|EXAMPLE BILLING LTD",
            "merchant_final": "EXAMPLE DOCTOR",
            "spending_category": "Health / Medication",
            "exclude_from_consumption": "",
            "fixed_variable": "",
            "note": "Online doctor billed via a generic company name"
        }
    ],
    columns=OVERRIDE_COLUMNS
)

merchant_overrides_path = MANUAL_PATH / "merchant_overrides_manual.csv"

if not merchant_overrides_path.exists():
    merchant_overrides_example_seed.to_csv(merchant_overrides_path, index=False)
    print(f"Seeded example overrides: {merchant_overrides_path}")

merchant_overrides = pd.read_csv(merchant_overrides_path, dtype=str)

for col in OVERRIDE_COLUMNS:
    if col not in merchant_overrides.columns:
        merchant_overrides[col] = np.nan

print(f"Loaded {len(merchant_overrides)} merchant overrides.")

### 5.3 Classify transactions

In [ ]:
classification = consumption.apply(
    lambda row: classify_consumption_transaction(
        merchant_clean=row["merchant_clean"],
        description=row["description"],
        mcc_code=row["mcc_code"],
        overrides=merchant_overrides
    ),
    axis=1,
    result_type="expand"
)

consumption = pd.concat([consumption, classification], axis=1)

# Category-based rule, unless an override sets fixed_variable explicitly
consumption["fixed_variable"] = consumption["fixed_variable_override"].fillna(
    consumption.apply(
        lambda row: classify_fixed_variable_from_category(
            spending_category=row["spending_category"],
            merchant_final=row["merchant_final"],
            exclude_from_consumption=row["exclude_from_consumption"]
        ),
        axis=1
    )
)

consumption_analysis = consumption[
    ~consumption["exclude_from_consumption"]
].copy()

(
    consumption
    .groupby("classification_source")
    .agg(
        n_transactions=("spending_amount", "size"),
        total_spending=("spending_amount", "sum")
    )
    .sort_values("n_transactions", ascending=False)
)

### 5.4 Validation against legacy labels

`data/manual/consumption_legacy_labels.csv` (gitignored) is a snapshot of the final categories from the previous keyword + web-search + LLM + manual-review pipeline. Transactions the legacy pipeline left unclear are excluded from the agreement rate.

In [ ]:
legacy_labels_path = MANUAL_PATH / "consumption_legacy_labels.csv"

if legacy_labels_path.exists():
    legacy_labels = pd.read_csv(legacy_labels_path)

    validation = consumption.merge(
        legacy_labels[["transaction_id", "legacy_spending_category"]],
        on="transaction_id",
        how="inner"
    )

    validation_known = validation[
        ~validation["legacy_spending_category"].isin(["Uncategorized", "Other"])
    ].copy()

    validation_known["agrees"] = (
        validation_known["spending_category"]
        == validation_known["legacy_spending_category"]
    )

    print(f"Matched transactions: {len(validation)}")
    print(f"Legacy-labelled (excl. unclear): {len(validation_known)}")
    print(f"Agreement rate: {validation_known['agrees'].mean():.1%}")

    display(
        validation_known
        .groupby("classification_source")["agrees"]
        .agg(["size", "mean"])
    )

    legacy_disagreements = (
        validation_known[~validation_known["agrees"]]
        .groupby(
            [
                "merchant_clean",
                "mcc_code",
                "legacy_spending_category",
                "spending_category",
                "classification_source"
            ],
            dropna=False
        )
        .agg(
            n_transactions=("spending_amount", "size"),
            total_spending=("spending_amount", "sum")
        )
        .sort_values("total_spending", ascending=False)
        .reset_index()
    )

    display(legacy_disagreements)
else:
    print("No legacy labels found -- skipping validation.")

### 5.5 Final Unclear Merchant Audit

In [ ]:
final_unclear = consumption_analysis[
    consumption_analysis["spending_category"].isin(["Other", "Uncategorized"])
    | consumption_analysis["spending_category"].isna()
].copy()

final_unclear_summary = (
    final_unclear
    .groupby("merchant_clean", dropna=False)
    .agg(
        total_spending=("spending_amount", "sum"),
        n_transactions=("spending_amount", "size"),
        merchant_final=("merchant_final", "first"),
        spending_category=("spending_category", "first"),
        fixed_variable=("fixed_variable", "first"),
        example_names=("merchant_raw", preview_unique),
        example_descriptions=("description", preview_unique)
    )
    .sort_values("total_spending", ascending=False)
    .reset_index()
)

final_unclear_summary.head(100)

## 6. Merchant Spending Summary

In [ ]:
merchant_summary = (
    consumption_analysis
    .groupby("merchant_final", dropna=False)
    .agg(
        total_spending=("spending_amount", "sum"),
        n_transactions=("spending_amount", "size"),
        avg_transaction=("spending_amount", "mean"),
        median_transaction=("spending_amount", "median"),
        first_date=("date", "min"),
        last_date=("date", "max")
    )
    .sort_values("total_spending", ascending=False)
    .reset_index()
)

merchant_summary.head(30)

In [ ]:
top_merchants = merchant_summary.head(15).copy()

plt.figure(figsize=(10, 6))

plt.barh(
    top_merchants["merchant_final"][::-1],
    top_merchants["total_spending"][::-1]
)

plt.title("Top Merchants by Total Spending")
plt.xlabel("Total Spending (€)")
plt.ylabel("")
plt.tight_layout()
plt.show()

## 7. Monthly Merchant Trends

In [ ]:
merchant_monthly = (
    consumption_analysis
    .groupby(["month", "merchant_final"], dropna=False)
    .agg(
        monthly_spending=("spending_amount", "sum"),
        n_transactions=("spending_amount", "size")
    )
    .reset_index()
)

merchant_monthly.head()

In [ ]:
top_merchant_names = merchant_summary.head(8)["merchant_final"].tolist()

top_merchant_monthly = merchant_monthly[
    merchant_monthly["merchant_final"].isin(top_merchant_names)
].copy()

plt.figure(figsize=(12, 6))

for merchant in top_merchant_names:
    temp = top_merchant_monthly[
        top_merchant_monthly["merchant_final"] == merchant
    ]

    plt.plot(
        temp["month"],
        temp["monthly_spending"],
        marker="o",
        label=merchant
    )

plt.title("Monthly Spending for Top Merchants")
plt.xlabel("Month")
plt.ylabel("Spending (€)")
plt.legend()
plt.xticks(rotation=45)
plt.tight_layout()
plt.show()

## 8. Category Spending Summary

In [ ]:
category_summary = (
    consumption_analysis
    .groupby("spending_category", dropna=False)
    .agg(
        total_spending=("spending_amount", "sum"),
        n_transactions=("spending_amount", "size"),
        avg_transaction=("spending_amount", "mean"),
        median_transaction=("spending_amount", "median")
    )
    .sort_values("total_spending", ascending=False)
    .reset_index()
)

category_summary

In [ ]:
plt.figure(figsize=(10, 5))

plt.barh(
    category_summary["spending_category"][::-1],
    category_summary["total_spending"][::-1]
)

plt.title("Spending by Category")
plt.xlabel("Total Spending (€)")
plt.ylabel("")
plt.tight_layout()
plt.show()

## 9. Recurring Detection

In [ ]:
merchant_monthly_full = (
    consumption_analysis
    .groupby(["merchant_final", "month"], dropna=False)
    .agg(
        monthly_spending=("spending_amount", "sum"),
        n_transactions=("spending_amount", "size")
    )
    .reset_index()
)

recurring_summary = (
    merchant_monthly_full
    .groupby("merchant_final", dropna=False)
    .agg(
        active_months=("month", "nunique"),
        first_month=("month", "min"),
        last_month=("month", "max"),
        avg_monthly_spending=("monthly_spending", "mean"),
        median_monthly_spending=("monthly_spending", "median"),
        std_monthly_spending=("monthly_spending", "std"),
        total_spending=("monthly_spending", "sum"),
        max_monthly_spending=("monthly_spending", "max")
    )
    .reset_index()
)

recurring_summary["cv_monthly_spending"] = (
    recurring_summary["std_monthly_spending"]
    / recurring_summary["avg_monthly_spending"]
)

recurring_summary["cv_monthly_spending"] = (
    recurring_summary["cv_monthly_spending"]
    .replace([np.inf, -np.inf], np.nan)
)

# Months between first and last occurrence (inclusive). Without this, three
# purchases spread over a year would count as "recurring".
first_month = pd.to_datetime(recurring_summary["first_month"])
last_month = pd.to_datetime(recurring_summary["last_month"])

recurring_summary["span_months"] = (
    (last_month.dt.year - first_month.dt.year) * 12
    + (last_month.dt.month - first_month.dt.month)
    + 1
)

recurring_summary["active_share"] = (
    recurring_summary["active_months"] / recurring_summary["span_months"]
)

recurring_summary["likely_recurring_payment"] = (
    (recurring_summary["active_months"] >= 3)
    & (recurring_summary["active_share"] >= 0.75)
    & (recurring_summary["cv_monthly_spending"] <= 0.50)
    & (recurring_summary["avg_monthly_spending"] >= 3)
)

recurring_summary = recurring_summary.sort_values(
    ["likely_recurring_payment", "total_spending"],
    ascending=[False, False]
)

recurring_summary.head(40)

## 10. Fixed vs Variable Spending

In [ ]:
fixed_variable_summary = (
    consumption_analysis
    .groupby("fixed_variable", dropna=False)
    .agg(
        total_spending=("spending_amount", "sum"),
        n_transactions=("spending_amount", "size"),
        avg_transaction=("spending_amount", "mean"),
        median_transaction=("spending_amount", "median")
    )
    .sort_values("total_spending", ascending=False)
    .reset_index()
)

fixed_variable_summary

In [ ]:
plt.figure(figsize=(8, 4))

plt.barh(
    fixed_variable_summary["fixed_variable"][::-1],
    fixed_variable_summary["total_spending"][::-1]
)

plt.title("Fixed vs Variable Spending")
plt.xlabel("Total Spending (€)")
plt.ylabel("")
plt.tight_layout()
plt.show()

## 11. Large Transactions / Outliers

In [ ]:
large_transactions = (
    consumption_analysis
    .sort_values("spending_amount", ascending=False)
    [
        [
            "date",
            "merchant_clean",
            "merchant_final",
            "spending_category",
            "fixed_variable",
            "spending_amount",
            "description"
        ]
    ]
    .head(50)
)

large_transactions

In [ ]:
outlier_threshold = consumption_analysis["spending_amount"].quantile(0.95)

outlier_transactions = consumption_analysis[
    consumption_analysis["spending_amount"] >= outlier_threshold
].copy()

outlier_transactions = outlier_transactions.sort_values(
    "spending_amount",
    ascending=False
)

print(f"95th percentile threshold: {outlier_threshold:.2f} €")
print(f"Number of outlier transactions: {len(outlier_transactions)}")

outlier_transactions[
    [
        "date",
        "merchant_final",
        "spending_category",
        "fixed_variable",
        "spending_amount",
        "description"
    ]
].head(50)

## 12. Monthly Category Trends

In [ ]:
category_monthly = (
    consumption_analysis
    .groupby(["month", "spending_category"], dropna=False)
    .agg(
        monthly_spending=("spending_amount", "sum"),
        n_transactions=("spending_amount", "size")
    )
    .reset_index()
)

category_monthly.head()

In [ ]:
consumption_monthly_summary = (
    consumption_analysis
    .groupby("month", dropna=False)
    .agg(
        total_spending=("spending_amount", "sum"),
        n_transactions=("spending_amount", "size"),
        avg_transaction=("spending_amount", "mean"),
        median_transaction=("spending_amount", "median"),
        n_merchants=("merchant_final", "nunique"),
        n_categories=("spending_category", "nunique")
    )
    .reset_index()
    .sort_values("month")
)

consumption_monthly_summary

In [ ]:
top_categories = (
    category_summary
    .head(8)["spending_category"]
    .tolist()
)

top_category_monthly = category_monthly[
    category_monthly["spending_category"].isin(top_categories)
].copy()

plt.figure(figsize=(12, 6))

for category in top_categories:
    temp = top_category_monthly[
        top_category_monthly["spending_category"] == category
    ]

    plt.plot(
        temp["month"],
        temp["monthly_spending"],
        marker="o",
        label=category
    )

plt.title("Monthly Spending by Top Categories")
plt.xlabel("Month")
plt.ylabel("Spending (€)")
plt.legend()
plt.xticks(rotation=45)
plt.tight_layout()
plt.show()

## 13. Dashboard Exports

In [ ]:
export_path = DATA_PATH

consumption_analysis.to_csv(
    export_path / "consumption_transactions_enriched.csv",
    index=False
)

merchant_summary.to_csv(
    export_path / "consumption_merchant_summary.csv",
    index=False
)

merchant_monthly.to_csv(
    export_path / "consumption_merchant_monthly.csv",
    index=False
)

category_summary.to_csv(
    export_path / "consumption_category_summary.csv",
    index=False
)

category_monthly.to_csv(
    export_path / "consumption_category_monthly.csv",
    index=False
)

consumption_monthly_summary.to_csv(
    export_path / "consumption_monthly_summary.csv",
    index=False
)

fixed_variable_summary.to_csv(
    export_path / "consumption_fixed_variable_summary.csv",
    index=False
)

recurring_summary.to_csv(
    export_path / "consumption_recurring_summary.csv",
    index=False
)

large_transactions.to_csv(
    export_path / "consumption_large_transactions.csv",
    index=False
)

final_unclear_summary.to_csv(
    export_path / "consumption_final_unclear_summary.csv",
    index=False
)

print("Saved consumption analytics exports to:")
print(export_path)

In [ ]:
consumption_kpis = pd.DataFrame(
    [
        {
            "total_consumption": consumption_analysis["spending_amount"].sum(),
            "n_consumption_transactions": len(consumption_analysis),
            "avg_transaction": consumption_analysis["spending_amount"].mean(),
            "median_transaction": consumption_analysis["spending_amount"].median(),
            "n_merchants": consumption_analysis["merchant_final"].nunique(),
            "n_categories": consumption_analysis["spending_category"].nunique(),
            "unclear_total_spending": final_unclear_summary["total_spending"].sum()
                if len(final_unclear_summary) > 0 else 0,
            "n_unclear_merchants": len(final_unclear_summary)
        }
    ]
)

consumption_kpis.to_csv(
    export_path / "consumption_kpis.csv",
    index=False
)

consumption_kpis

## 14. Final Checks and Key Takeaways

In [ ]:
print("Consumption transactions:", len(consumption_analysis))
print("Total consumption:", round(consumption_analysis["spending_amount"].sum(), 2))
print("Unique merchants:", consumption_analysis["merchant_final"].nunique())
print("Unique categories:", consumption_analysis["spending_category"].nunique())
print("Final unclear merchants:", len(final_unclear_summary))

print("\nTop categories:")
display(category_summary.head(10))

print("\nTop merchants:")
display(merchant_summary.head(10))

print("\nFixed / Variable:")
display(fixed_variable_summary)

print("\nFinal unclear:")
display(final_unclear_summary.head(30))